# Lesson 32 — Exercises

## Exercise 1 — Dominance by hand (10 pts)

Eight pump designs are scored on **cost** (k\$, minimize) and **reliability** (maximize):

| design | A | B | C | D | E | F | G | H |
|---|---|---|---|---|---|---|---|---|
| cost | 10 | 12 | 11 | 15 | 9 | 14 | 12 | 9 |
| reliability | 0.90 | 0.95 | 0.89 | 0.95 | 0.80 | 0.99 | 0.93 | 0.85 |

1. **By hand**, list the nondominated designs. For each dominated design, name one design that dominates it.
2. Give the ideal and nadir points of the front, in the original senses.
3. Check your answer with `discopt.mo.filter_nondominated` (pass `senses`).

**Expected:** front $=\{A, B, F, H\}$, ideal $=(9,\,0.99)$, nadir $=(14,\,0.85)$. Notice that $E$ is dominated by $H$ even though they cost the same.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
# TODO
from discopt.mo import filter_nondominated

labels = list("ABCDEFGH")
P = np.array([[10, 0.90], [12, 0.95], [11, 0.89], [15, 0.95],
              [9, 0.80], [14, 0.99], [12, 0.93], [9, 0.85]])

my_front = None   # e.g. {"A", ...}: your hand answer
my_ideal = None   # (cost, reliability)
my_nadir = None   # (cost, reliability)

# mask = filter_nondominated(P, senses=...)   # TODO

if my_front is not None:
    assert my_front == {"A", "B", "F", "H"}
    assert np.allclose(my_ideal, (9, 0.99)) and np.allclose(my_nadir, (14, 0.85))


## Exercise 2 — $\varepsilon$-constraint duals as a carbon price (15 pts)

A utility must supply 100 MWh from coal, gas and wind:

| | coal | gas | wind |
|---|---|---|---|
| cost (\$/MWh) | 20 | 35 | 60 |
| emissions (t/MWh) | 1.0 | 0.5 | 0.0 |
| capacity (MWh) | 60 | 50 | 40 |

1. Build $P(\varepsilon)$: $\min$ cost s.t. demand $=100$, emissions $\le \varepsilon$. Make $\varepsilon$ an `m.parameter` and name the row (`name="emis"`). Sweep $\varepsilon$ from the emissions anchor to the cost anchor.
2. At each $\varepsilon$, record $z^\star$, $\lambda=$ `r.constraint_duals["emis"]` and $-$`r.gradient(eps)` (solve with `sensitivity=True`). Plot $z^\star(\varepsilon)$ and mark the breakpoints. Explain each slope with a **fuel switch**: which source replaces which, and why the cost per tonne avoided is $\Delta c / \Delta e$.
3. Now drop the $\varepsilon$ row and minimize cost $+\;p\cdot$ emissions for carbon prices $p \in \{10, 35, 45\}$. Which front point does each price select? Connect this to $\lambda$ (Reading §4).
4. Cross-check the whole front with `discopt.mo.epsilon_constraint`.

**Expected:** anchors $\varepsilon \in [35, 80]$ with $z^\star(35)=4350$ and $z^\star(80)=2600$. $\lambda = 30$ \$/t on $(75, 80)$ (coal → gas) and $\lambda = 40$ \$/t on $(35, 75)$ (coal → wind). $z^\star(50) = 3750$, $\lambda(50)=40$, $\lambda(78)=30$. Prices $p=10, 35, 45$ select emissions $80, 75, 35$.

In [ ]:
# TODO
cost = np.array([20.0, 35.0, 60.0])
em   = np.array([1.0, 0.5, 0.0])
cap  = np.array([60.0, 50.0, 40.0])

z_at  = {}   # eps -> z*(eps), e.g. z_at[50.0] = ...
lam_at = {}  # eps -> lambda from r.constraint_duals["emis"]

if 50.0 in z_at and 78.0 in lam_at:
    assert abs(z_at[50.0] - 3750) < 1e-3
    assert abs(lam_at[50.0] - 40) < 1e-3 and abs(lam_at[78.0] - 30) < 1e-3


## Exercise 3 — Supported vs unsupported points in a bi-objective knapsack (15 pts)

Choose a subset of 8 projects under a budget $\sum_i w_i x_i \le 18$, and **maximize** both profit $p^\top x$ and jobs created $q^\top x$:

```
p = [12,  9, 15,  7, 11,  6, 14,  8]
q = [ 4, 10,  3,  9,  6, 12,  5,  8]
w = [ 5,  4,  7,  3,  5,  3,  6,  4]
```

1. Enumerate all $2^8$ subsets to get the exact nondominated set.
2. Run `weighted_sum(..., senses=["max", "max"], n_weights=51)`. Which points does it miss? Show on a plot that the missed points lie *inside* the convex hull of the front. These are the unsupported points {cite:p}`ehrgott-2005`.
3. Run `epsilon_constraint` twice: once with `n_points=6`, and once with a grid step $\le 1$ in jobs. The jobs are integer, so this grid is exact. How many points does each run find, and why?

**Expected:** 7 nondominated points, from $(30,39)$ to $(42,28)$. Of these, 5 are supported. The weighted sum misses the unsupported $(33,37)$ and $(37,35)$, even with 51 weights. AUGMECON2 with `n_points=12` (step 1 over jobs $\in[28,39]$) finds all 7. With `n_points=6` it finds 6 and misses $(36,36)$.

In [ ]:
# TODO
p = np.array([12, 9, 15, 7, 11, 6, 14, 8])
q = np.array([4, 10, 3, 9, 6, 12, 5, 8])
w = np.array([5, 4, 7, 3, 5, 3, 6, 4])
W = 18

exact_front = None    # (n, 2) array of nondominated (profit, jobs)
ws_points = None      # (m, 2) array returned by weighted_sum

if exact_front is not None and ws_points is not None:
    assert len(exact_front) == 7 and len(ws_points) == 5
    missed = {tuple(r) for r in np.round(exact_front).astype(int)} - \
             {tuple(r) for r in np.round(ws_points).astype(int)}
    assert missed == {(33, 37), (37, 35)}


## Exercise 4 — Quality indicators on a concave front (15 pts)

Use the quarter-circle problem from Reading §3 ($\min x$, $\min y$, $x^2+y^2 \ge 1$, $x,y\in[0,1]$).

1. Generate fronts with `weighted_sum`, `weighted_tchebycheff`, `normal_boundary_intersection` and `epsilon_constraint`, 11 points or weights each. Build a fresh model for every call.
2. Build a reference front from 201 analytic points $(\cos\theta, \sin\theta)$ as a `ParetoFront` of `ParetoPoint`s (`x={}`, `status="analytic"`, `wall_time=0.0`).
3. With the **shared** reference point $r=(1.1, 1.1)$, tabulate hypervolume, `igd` (against the analytic front), `spread` and `epsilon_indicator`.
4. What is the hypervolume of the *true* front with respect to $r$, in closed form? Why is every finite front below it? Which indicator would rank the methods differently from hypervolume, and why is that misleading?

**Expected (±0.005):** HV of weighted sum $= 0.210$, Tchebycheff $\approx 0.384$, NBI $\approx 0.390$, AUGMECON2 $\approx 0.384$. Dense analytic front $\approx 0.4226$. Exact limit $1.21 - \pi/4 \approx 0.4246$. IGD: weighted sum $\approx 0.386$, the other three $\approx 0.04$–$0.06$. The weighted sum has the *best* spread ($0$, two points).

In [ ]:
# TODO
from discopt.mo import (weighted_sum, weighted_tchebycheff, normal_boundary_intersection,
                        epsilon_constraint, ParetoFront, ParetoPoint, hypervolume, igd,
                        spread, epsilon_indicator)

def build_concave():
    m = dm.Model("concave")
    x = m.continuous("x", lb=0, ub=1)
    y = m.continuous("y", lb=0, ub=1)
    m.subject_to(x**2 + y**2 >= 1.0)
    return m, [x, y]

r = np.array([1.1, 1.1])
hv = {}   # method name -> hypervolume, e.g. hv["weighted_sum"] = ...

if hv:
    assert abs(hv["weighted_sum"] - 0.21) < 5e-3
    assert hv["normal_boundary_intersection"] > hv["weighted_sum"] + 0.15


## Exercise 5 — Choosing a compromise facility plan (15 pts)

Use the facility location model of Reading §6 with this instance (5 sites, 7 customers):

```python
rng = np.random.default_rng(7)
nS, nC = 5, 7
sites, cust = rng.uniform(0, 10, (nS, 2)), rng.uniform(0, 10, (nC, 2))
dist = np.linalg.norm(sites[:, None] - cust[None], axis=2)
d = rng.integers(5, 15, nC).astype(float)
phi = np.array([30.0, 40.0, 55.0, 70.0, 90.0]); e = np.array([3.0, 2.4, 1.6, 1.0, 0.4])
u = np.full(nS, 35.0)
```

1. Enumerate the $5^7$ single-sourcing assignments to get the exact front.
2. Run `epsilon_constraint(..., n_points=41)`. Which exact points are missing, and why? (Look at the $\varepsilon$ step against the smallest gap in emissions between neighbouring Pareto points.) Choose `n_points` so that the sweep is complete, and verify that it is.
3. Normalize with the ideal and nadir of the complete front. Report the compromise chosen by (a) the $L_2$ distance to the ideal, (b) the $L_\infty$ distance to the ideal, and (c) `weighted_tchebycheff(..., weights=np.array([[0.5, 0.5]]), ideal=..., nadir=..., anchors=[])`. For each one, list the **open sites**.
4. Does the $L_\infty$ compromise change between the 41-point and the complete front? What does that say about deciding from an incomplete front?

**Expected:** 14 exact Pareto points, ideal $(183.85, 45.0)$, nadir $(363.40, 179.4)$. With 41 points, AUGMECON2 returns 11 and misses $(195.21, 178.2)$, $(218.95, 177.6)$ and $(267.56, 109.6)$. `n_points=225` (step $0.6$) recovers all 14. $L_2$ compromise: $(272.85, 88.4)$, sites $\{1, 4\}$. $L_\infty$ compromise: $(272.85, 88.4)$ on the 41-point front, but $(267.56, 109.6)$ with sites $\{0, 4\}$ on the complete front. Tchebycheff with equal weights returns $(267.56, 109.6)$ in a single solve.

In [ ]:
# TODO
rng = np.random.default_rng(7)
nS, nC = 5, 7
sites, cust = rng.uniform(0, 10, (nS, 2)), rng.uniform(0, 10, (nC, 2))
dist = np.linalg.norm(sites[:, None] - cust[None], axis=2)
d = rng.integers(5, 15, nC).astype(float)
phi = np.array([30.0, 40.0, 55.0, 70.0, 90.0]); e = np.array([3.0, 2.4, 1.6, 1.0, 0.4])
u = np.full(nS, 35.0)

n_exact = None        # number of exact Pareto points
linf_choice = None    # (cost, emissions) of the L_inf compromise on the complete front
linf_sites = None     # set of open site indices for that plan

if n_exact is not None:
    assert n_exact == 14
    assert np.allclose(linf_choice, (267.558, 109.6), atol=1e-2) and linf_sites == {0, 4}
